In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc samplomatic
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Inicio rápido con Executor

<Accordion>
<AccordionItem title="Versiones de paquetes">

El código de esta página fue desarrollado con los siguientes requisitos.
Recomendamos usar estas versiones o más recientes.

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
samplomatic~=0.18.0
```
</AccordionItem>
</Accordion>

De manera similar al primitivo [Sampler](/guides/get-started-with-sampler), Executor muestrea registros de salida de ejecuciones de circuits cuánticos, pero no tiene supresión ni mitigación de errores integradas. En su lugar, forma parte del [modelo de ejecución dirigida](/guides/directed-execution-model) que proporciona los ingredientes para capturar las intenciones de diseño en el lado del cliente y desplaza la costosa generación de variantes de circuits al lado del servidor. Executor sigue las directivas proporcionadas en las anotaciones y opciones del circuit, genera y vincula valores de parámetros, ejecuta los circuits vinculados en el hardware y devuelve los resultados de ejecución y los metadatos. No toma ninguna decisión implícita por ti y te da control total y transparencia.

> **Note:** El paquete Qiskit aún no tiene una clase base para el primitivo Executor.

## Antes de comenzar
Algunos de los ejemplos de código en esta página usan `samplex`, que forma parte del paquete Samplomatic. Por lo tanto, antes de ejecutar esos bloques de código, debes instalar Samplomatic, como se muestra en el siguiente bloque de código. Para más información, consulta la [documentación de Samplomatic](https://qiskit.github.io/samplomatic).

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService, Executor
from qiskit_ibm_runtime.quantum_program import QuantumProgram
from qiskit.circuit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager
from samplomatic.transpiler import generate_boxing_pass_manager
from samplomatic import build

# Initialize the service and choose a backend
service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

In [2]:
print(backend)

<IBMBackend('ibm_fez')>


### 2. Create and transpile a circuit

You need at least one circuit to use the Executor primitive.  It can optionally have parameters.

In [3]:
# Generate the circuit
circuit = QuantumCircuit(2)
circuit.h(0)
circuit.h(1)
circuit.cz(0, 1)
circuit.h(1)

# Using `measure_all` automatically creates the necessary
# classical registers.
circuit.measure_all()

The circuit needs to be transformed to only use instructions supported by the QPU (referred to as *instruction set architecture (ISA)* circuits). Use the transpiler to do this.

In [4]:
# Transpile the circuit
preset_pass_manager = generate_preset_pass_manager(
    backend=backend, optimization_level=0
)
isa_circuit = preset_pass_manager.run(circuit)

### 2. Crear y transpilar un circuit
Necesitas al menos un circuit para usar el primitivo Executor. Opcionalmente puede tener parámetros.

In [5]:
# Initialize an empty program
program = QuantumProgram(shots=25)

# Append the circuit to the program
program.append_circuit_item(isa_circuit)

El circuit debe transformarse para usar solo instrucciones admitidas por la QPU (denominadas circuits de *arquitectura de conjunto de instrucciones (ISA)*). Usa el transpilador para hacer esto.

In [6]:
# Generate a boxing pass manager to group gates
# and measurements into boxes and add
# a`Twirl` annotation.
boxes_pm = generate_boxing_pass_manager(
    # Add gate twirling
    enable_gates=True,
    # Add measurement twirling
    enable_measures=True,
)

boxed_circuit = boxes_pm.run(isa_circuit)
boxed_circuit.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/get-started-with-executor/extracted-outputs/245a4574-3ce9-4f77-98c8-af32cde8ac01-0.svg" alt="Output of the previous code cell" />

### 3. Inicializar un `QuantumProgram`
Inicializa un `QuantumProgram` con tu carga de trabajo. Un `QuantumProgram` está compuesto de `QuantumProgramItems`. Típicamente, cada elemento consiste en un circuit, un conjunto de valores de parámetros y posiblemente un `samplex` para aleatorizar el contenido del circuit. Para más detalles, consulta [Entradas y salidas de Executor](/guides/executor-input-output).

La siguiente celda inicializa un `QuantumProgram` y especifica realizar 25 shots. Luego, agrega el circuit transpilado objetivo.

In [7]:
# Build the template circuit and the samplex
template_circuit, samplex = build(boxed_circuit)

# Append the template circuit and samplex as a `samplex_item`
program.append_samplex_item(
    template_circuit,
    samplex=samplex,
    shape=(num_randomizations := 20,),
)

### 4. Opcional: Agrupar gates y mediciones en cajas anotadas
Agrupar instrucciones en cajas y anotarlas es la forma principal de especificar tu intención. En el siguiente ejemplo, usamos `generate_boxing_pass_manager` y sus parámetros de twirling para agrupar gates de dos qubits y mediciones en cajas y aplicar anotación de twirling.

In [8]:
# Initialize an Executor with the default options
executor = Executor(mode=backend)

# Submit the job
job = executor.run(program)
job

<RuntimeJobV2('d8286580bvlc73d1vmsg', 'executor')>

In [9]:
# Retrieve the result
result = job.result()

### 6. Invocar Executor y obtener resultados
Ejecuta el `QuantumProgram` en un backend de IBM&reg; usando el primitivo `Executor` con opciones predeterminadas. Consulta [Opciones de Executor](/guides/executor-options) para aprender sobre las opciones disponibles.